# 2. Prepare dataset for training

In this notebook, we will prepare the dataset for training with language models.

The preparation includes:
- Dropping irrelevant columns
- Adding toxicity difference metric
- Analyzing text length distribution

In [1]:
import pandas as pd

In [2]:
# Reading the data from the file
df = pd.read_csv("../data/raw/paranmt_for_detox_500k.tsv", sep="\t", index_col=0)
df_rows = df.shape[0]

df.head()

,reference,translation,similarity,length_diff,ref_tox,trn_tox
0,"If Alkar is flooding her with psychic waste, t...","if Alkar floods her with her mental waste, it ...",0.785171,0.010309,0.014195,0.981983
1,Now you're getting nasty.,you're becoming disgusting.,0.749687,0.071429,0.065473,0.999039
2,"Well, we could spare your life, for one.","well, we can spare your life.",0.919051,0.268293,0.213313,0.985068
3,"Ah! Monkey, you've got to snap out of it.","monkey, you have to wake up.",0.664333,0.309524,0.053362,0.994215
4,I've got orders to put her down.,I have orders to kill her.,0.726639,0.181818,0.009402,0.999348


In [3]:
MAX_TEXT_LENGTH = 700

In [4]:
# Drop length_diff column
df = df.drop(columns=["length_diff"])

# Drop rows with similarity less than 0.5
df = df[df["similarity"] >= 0.5]

In [5]:
# Drop rows with reference or translation text length more than MAX_TEXT_LENGTH symbols
df = df[df["reference"].str.len() <= MAX_TEXT_LENGTH]
df = df[df["translation"].str.len() <= MAX_TEXT_LENGTH]

In [6]:
# Get the longest reference and translation texts
print(f"Longest reference text: {df['reference'].str.len().max()}")
print(f"Longest translation text: {df['translation'].str.len().max()}")

Longest reference text: 677
Longest translation text: 688


In [7]:
# Add toxicity difference column
df["tox_diff"] = df["ref_tox"] - df["trn_tox"]

# Drop rows with toxicity difference absolute value less than 0.5
df = df[df["tox_diff"].abs() >= 0.5]

df.head()

,reference,translation,similarity,ref_tox,trn_tox,tox_diff
0,"If Alkar is flooding her with psychic waste, t...","if Alkar floods her with her mental waste, it ...",0.785171,0.014195,0.981983,-0.967788
1,Now you're getting nasty.,you're becoming disgusting.,0.749687,0.065473,0.999039,-0.933567
2,"Well, we could spare your life, for one.","well, we can spare your life.",0.919051,0.213313,0.985068,-0.771755
3,"Ah! Monkey, you've got to snap out of it.","monkey, you have to wake up.",0.664333,0.053362,0.994215,-0.940853
4,I've got orders to put her down.,I have orders to kill her.,0.726639,0.009402,0.999348,-0.989946


In [8]:
# Count overall number of dropped rows
print(f"Dropped overall {df_rows - df.shape[0]} rows")

Dropped overall 24 rows


In [9]:
# Prepare for packing back to file
# First, let's swap reference and translation texts if the translation is more toxic
index_mask = df["tox_diff"] < 0

df.loc[index_mask, "reference"], df.loc[index_mask, "translation"] = (
    df.loc[index_mask, "translation"],
    df.loc[index_mask, "reference"],
)

# ..and change the toxicity difference sign
df.loc[index_mask, "tox_diff"] = -df.loc[index_mask, "tox_diff"]
df.head()

,reference,translation,similarity,ref_tox,trn_tox,tox_diff
0,"if Alkar floods her with her mental waste, it ...","If Alkar is flooding her with psychic waste, t...",0.785171,0.014195,0.981983,0.967788
1,you're becoming disgusting.,Now you're getting nasty.,0.749687,0.065473,0.999039,0.933567
2,"well, we can spare your life.","Well, we could spare your life, for one.",0.919051,0.213313,0.985068,0.771755
3,"monkey, you have to wake up.","Ah! Monkey, you've got to snap out of it.",0.664333,0.053362,0.994215,0.940853
4,I have orders to kill her.,I've got orders to put her down.,0.726639,0.009402,0.999348,0.989946


In [10]:
# Save only reference and translation texts
df[["reference", "translation"]].to_csv(
    "../data/interim/processed.tsv", sep="\t", index=False, header=False
)